# EA1 - Actividad 1.2: RDDs (Resilient Distributed Datasets)

## Objetivos
- Entender que son los RDDs y sus propiedades fundamentales
- Diferenciar entre transformaciones (lazy) y acciones
- Aplicar operaciones basicas: map, filter, flatMap, reduce, reduceByKey
- Comparar el rendimiento de RDDs vs DataFrames

## Conceptos Clave

### Que son los RDDs?

Los **RDD (Resilient Distributed Datasets)** son la estructura de datos fundamental de Spark. Son una coleccion distribuida e inmutable de elementos que se pueden procesar en paralelo.

**Propiedades principales:**

| Propiedad | Descripcion |
|-----------|-------------|
| **Inmutabilidad** | Una vez creado, un RDD no se puede modificar. Cada transformacion crea un nuevo RDD |
| **Distribuido** | Los datos se dividen en **particiones** que se distribuyen entre los nodos del cluster |
| **Resiliente** | Si una particion se pierde, Spark puede reconstruirla gracias al **lineage** (historial de transformaciones) |
| **Evaluacion lazy** | Las transformaciones no se ejecutan inmediatamente; se acumulan y se ejecutan cuando se invoca una accion |

### Transformaciones vs Acciones

- **Transformaciones:** Crean un nuevo RDD a partir de uno existente. Son **lazy** (no se ejecutan de inmediato).
  - Ejemplos: `map()`, `filter()`, `flatMap()`, `reduceByKey()`, `groupByKey()`

- **Acciones:** Devuelven un resultado al driver o escriben datos. **Disparan la ejecucion** de todas las transformaciones pendientes.
  - Ejemplos: `collect()`, `count()`, `take()`, `reduce()`, `first()`, `saveAsTextFile()`

In [ ]:
# Setup: Crear SparkSession y obtener SparkContext
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("02_rdds_basico") \
    .master("local[*]") \
    .getOrCreate()

# SparkContext es el punto de entrada para trabajar con RDDs
sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print(f"SparkContext: {sc}")

## Crear RDDs

Hay dos formas principales de crear un RDD:
1. **Desde una coleccion en memoria** con `sc.parallelize()`
2. **Desde un archivo externo** con `sc.textFile()`

In [ ]:
# Crear un RDD desde una lista de Python
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print(f"Tipo: {type(rdd)}")
print(f"Numero de particiones: {rdd.getNumPartitions()}")

## Transformaciones (Lazy)

Las transformaciones crean un nuevo RDD pero **no se ejecutan** hasta que se invoque una accion.

### `map()` - Aplica una funcion a cada elemento

In [ ]:
# map: multiplicar cada elemento por 2
rdd_doble = rdd.map(lambda x: x * 2)

# Nota: hasta aqui NO se ha ejecutado nada!
# Solo se ha registrado la transformacion
print(f"rdd_doble es de tipo: {type(rdd_doble)}")
print("La transformacion aun no se ejecuto (lazy evaluation)")

### `filter()` - Selecciona elementos que cumplen una condicion

In [ ]:
# filter: obtener solo numeros mayores a 5
rdd_mayores = rdd.filter(lambda x: x > 5)

# Encadenar transformaciones: multiplicar por 2 Y luego filtrar > 10
rdd_encadenado = rdd.map(lambda x: x * 2).filter(lambda x: x > 10)

print("Transformaciones registradas (aun no ejecutadas)")

## Acciones (Disparan la ejecucion)

Las acciones fuerzan la evaluacion de todas las transformaciones pendientes.

In [ ]:
# collect(): devuelve TODOS los elementos como lista de Python
# CUIDADO: no usar con datasets grandes (todo va a memoria del driver)
print(f"Original: {rdd.collect()}")
print(f"Doble: {rdd_doble.collect()}")
print(f"Mayores a 5: {rdd_mayores.collect()}")
print(f"Encadenado (x2, >10): {rdd_encadenado.collect()}")

In [ ]:
# count(): cuenta el numero de elementos
print(f"Cantidad de elementos: {rdd.count()}")

# take(n): devuelve los primeros n elementos
print(f"Primeros 3: {rdd.take(3)}")

# reduce(): combina todos los elementos usando una funcion
suma = rdd.reduce(lambda a, b: a + b)
print(f"Suma total: {suma}")

## `flatMap()` - Aplanar resultados

A diferencia de `map()`, `flatMap()` puede devolver 0 o mas elementos por cada elemento de entrada, y los "aplana" en un solo RDD.

In [ ]:
# Ejemplo: dividir frases en palabras
frases = sc.parallelize([
    "Hola mundo de Spark",
    "Big Data es fascinante",
    "Spark procesa datos en paralelo"
])

# Con map: cada elemento se convierte en una lista de palabras
palabras_map = frases.map(lambda frase: frase.split(" "))
print(f"Con map (listas anidadas): {palabras_map.collect()}")

# Con flatMap: las listas se aplanan en elementos individuales
palabras_flat = frases.flatMap(lambda frase: frase.split(" "))
print(f"Con flatMap (plano): {palabras_flat.collect()}")

## Leer un archivo como RDD de texto

Con `sc.textFile()` cada linea del archivo se convierte en un elemento del RDD.

In [ ]:
# Leer flights.csv como RDD de lineas de texto
rdd_flights = sc.textFile("/home/jovyan/datos/flights.csv")

# Ver las primeras 3 lineas
for linea in rdd_flights.take(3):
    print(linea)

print(f"\nTotal de lineas (incluye header): {rdd_flights.count()}")

In [ ]:
# Obtener el header y separarlo de los datos
header = rdd_flights.first()
print(f"Header: {header}")

# Filtrar el header para quedarnos solo con los datos
rdd_datos = rdd_flights.filter(lambda linea: linea != header)
print(f"Lineas de datos: {rdd_datos.count()}")

---
## Ejercicios

Ahora es tu turno de practicar con RDDs.

In [ ]:
# =============================================================
# EJERCICIO 1: Cuadrados de numeros pares
# =============================================================
# TODO: Usando el RDD 'rdd' (que contiene [1,2,3,...,10]):
#   1. Filtra solo los numeros pares (x % 2 == 0)
#   2. Calcula el cuadrado de cada numero par (x ** 2)
#   3. Recolecta el resultado con .collect() e imprimelo
#
# Resultado esperado: [4, 16, 36, 64, 100]
#
# Pista: rdd.filter(...).map(...).collect()

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Word Count (conteo de palabras)
# =============================================================
# TODO: Implementa el clasico "Word Count" usando RDDs.
#   Dado el siguiente RDD de frases:

rdd_frases = sc.parallelize([
    "spark es rapido y spark es potente",
    "big data con spark",
    "spark para big data"
])

# Pasos:
#   1. Usa flatMap para dividir cada frase en palabras individuales
#   2. Usa map para convertir cada palabra en un par (palabra, 1)
#   3. Usa reduceByKey para sumar los conteos de cada palabra
#   4. Recolecta e imprime el resultado
#
# Pista: rdd.flatMap(lambda x: x.split(" "))
#        .map(lambda palabra: (palabra, 1))
#        .reduceByKey(lambda a, b: a + b)
#        .collect()

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Vuelos con retraso mayor a 30 minutos
# =============================================================
# TODO: Usando el RDD 'rdd_datos' (flights.csv sin header):
#   1. Parsea cada linea dividiendo por coma: linea.split(",")
#   2. Filtra los vuelos donde DEPARTURE_DELAY > 30
#      (DEPARTURE_DELAY esta en la posicion/indice que corresponda
#       segun el header; revisa el header impreso arriba)
#   3. Cuenta cuantos vuelos cumplen la condicion
#
# IMPORTANTE: Los campos vienen como texto. Debes convertir a numero
# y manejar valores vacios. Usa try/except o verifica que el campo
# no este vacio antes de convertir.
#
# Pista:
# - Primero identifica la posicion de DEPARTURE_DELAY en el header
# - header.split(",") te dara la lista de nombres de columnas
# - Usa map para parsear y filter para la condicion

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **RDDs:** Estructura de datos fundamental de Spark — inmutable, distribuida, resiliente
2. **Transformaciones (lazy):** `map()`, `filter()`, `flatMap()` — no se ejecutan hasta que hay una accion
3. **Acciones:** `collect()`, `count()`, `take()`, `reduce()` — disparan la ejecucion
4. **Crear RDDs:** Desde listas (`sc.parallelize()`) o archivos (`sc.textFile()`)
5. **Word Count:** Patron clasico de Big Data usando `flatMap` + `map` + `reduceByKey`
6. **Parseo de archivos CSV:** Separar header, split por coma, conversion de tipos

---
## Desafio Extra (Opcional)

**Comparar rendimiento: RDD vs DataFrame**

Realiza la misma operacion (contar vuelos por aerolinea) usando RDD y DataFrame.
Usa `%%time` al inicio de cada celda para comparar el tiempo de ejecucion.

Pregunta: Cual es mas rapido? Por que crees que ocurre esa diferencia?
(Pista: investiga el optimizador Catalyst de Spark SQL)

In [ ]:
# =============================================================
# DESAFIO: Comparar RDD vs DataFrame
# =============================================================
# TODO:
# Parte A - Con RDD:
#   Usa rdd_datos para contar vuelos por aerolinea
#   (parsear lineas, extraer columna AIRLINE, usar map + reduceByKey)
#
# Parte B - Con DataFrame:
#   Lee flights.csv como DataFrame y usa groupBy("AIRLINE").count()
#
# Compara los tiempos con %%time

# Escribe tu codigo aqui:


In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")